# 📚 College RAG Chatbot — Data Ingestion & Chunking

This notebook handles the complete **data ingestion pipeline** for a college RAG chatbot:

1. Load `.txt`, `.pdf`, and `faq.json` files from the `data/` folder
2. Chunk documents using `RecursiveCharacterTextSplitter`
3. Generate embeddings via HuggingFace (`all-MiniLM-L6-v2`)
4. Store everything in **ChromaDB**
5. Test retrieval with a sample query

> **Note:** This notebook does NOT include any LLM answer generation — it focuses solely on ingestion, embedding, and vector storage.

---
## 1️⃣ Install Required Packages

In [ ]:
# Install all required packages for the RAG pipeline
%pip install langchain langchain-community langchain-chroma langchain-huggingface langchain-text-splitters pypdf chromadb python-dotenv

---
## 2️⃣ Import Required Libraries

In [ ]:
import os
import json
import glob

# LangChain core
from langchain_core.documents import Document

# Document loaders
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader

# Text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector store
from langchain_chroma import Chroma

print("✅ All libraries imported successfully.")

---
## 3️⃣ Define Data Folder Path

In [ ]:
# Path to the folder containing all source documents
DATA_FOLDER = "data"

# Verify the folder exists
if not os.path.isdir(DATA_FOLDER):
    raise FileNotFoundError(f"❌ Data folder '{DATA_FOLDER}' not found. Please create it and add your files.")

# List all files in the data folder for a quick sanity check
files = os.listdir(DATA_FOLDER)
print(f"📂 Data folder: {os.path.abspath(DATA_FOLDER)}")
print(f"📄 Files found ({len(files)}):")
for f in sorted(files):
    print(f"   • {f}")

---
## 4️⃣ Load All `.txt` Files (TextLoader)

Each `.txt` file is loaded using LangChain's `TextLoader`. We tag every document with:
- **source** — the filename
- **type** — `"website"`

In [ ]:
txt_docs = []  # Accumulator for all text-file documents

# Discover every .txt file in the data folder
txt_files = glob.glob(os.path.join(DATA_FOLDER, "*.txt"))

if not txt_files:
    print("⚠️  No .txt files found in the data folder.")
else:
    for filepath in sorted(txt_files):
        filename = os.path.basename(filepath)
        try:
            loader = TextLoader(filepath, encoding="utf-8")
            docs = loader.load()

            # Attach custom metadata to each loaded document
            for doc in docs:
                doc.metadata["source"] = filename
                doc.metadata["type"] = "website"

            txt_docs.extend(docs)
            print(f"   ✅ Loaded: {filename}  →  {len(docs)} document(s)")
        except Exception as e:
            print(f"   ❌ Error loading {filename}: {e}")

print(f"\n📝 Total TXT documents loaded: {len(txt_docs)}")

---
## 5️⃣ Load All `.pdf` Files (PyPDFLoader)

Each `.pdf` file is loaded page-by-page using `PyPDFLoader`. Metadata tags:
- **source** — the filename
- **type** — `"pdf"`

In [ ]:
pdf_docs = []  # Accumulator for all PDF documents

# Discover every .pdf file in the data folder
pdf_files = glob.glob(os.path.join(DATA_FOLDER, "*.pdf"))

if not pdf_files:
    print("⚠️  No .pdf files found in the data folder.")
else:
    for filepath in sorted(pdf_files):
        filename = os.path.basename(filepath)
        try:
            loader = PyPDFLoader(filepath)
            docs = loader.load()

            # Attach custom metadata to each page document
            for doc in docs:
                doc.metadata["source"] = filename
                doc.metadata["type"] = "pdf"

            pdf_docs.extend(docs)
            print(f"   ✅ Loaded: {filename}  →  {len(docs)} page(s)")
        except Exception as e:
            print(f"   ❌ Error loading {filename}: {e}")

print(f"\n📄 Total PDF documents (pages) loaded: {len(pdf_docs)}")

---
## 6️⃣ Load FAQ Data (`faq.json`)

Each FAQ entry (with keys `question`, `answer`, `source`) is converted into a LangChain `Document`.

**page_content** format:
```
Question: {question}
Answer: {answer}
```

In [ ]:
faq_docs = []  # Accumulator for FAQ documents

# Path to the FAQ JSON file
faq_path = os.path.join(DATA_FOLDER, "f&q.json")

if not os.path.isfile(faq_path):
    print(f"⚠️  FAQ file not found at: {faq_path}")
else:
    try:
        with open(faq_path, "r", encoding="utf-8") as f:
            faq_data = json.load(f)

        for item in faq_data:
            # Build page_content in the required format
            page_content = f"Question: {item['question']}\nAnswer: {item['answer']}"

            # Create a LangChain Document with metadata
            doc = Document(
                page_content=page_content,
                metadata={
                    "source": item.get("source", "FAQ"),
                    "type": "faq"
                }
            )
            faq_docs.append(doc)

        print(f"✅ Loaded {len(faq_docs)} FAQ entries from f&q.json")

        # Preview the first FAQ document
        if faq_docs:
            print(f"\n--- Sample FAQ Document ---")
            print(faq_docs[0].page_content)
            print(f"Metadata: {faq_docs[0].metadata}")

    except json.JSONDecodeError as e:
        print(f"❌ Error parsing faq.json: {e}")
    except Exception as e:
        print(f"❌ Error loading faq.json: {e}")

print(f"\n❓ Total FAQ documents loaded: {len(faq_docs)}")

---
## 7️⃣ Combine All Documents

Merge TXT, PDF, and FAQ documents into a single unified list.

In [ ]:
# Combine all document lists into one
all_documents = txt_docs + pdf_docs + faq_docs

print(f"📊 Document Breakdown:")
print(f"   • TXT documents : {len(txt_docs)}")
print(f"   • PDF documents : {len(pdf_docs)}")
print(f"   • FAQ documents : {len(faq_docs)}")
print(f"   ─────────────────────────")
print(f"   📦 Total documents loaded: {len(all_documents)}")

---
## 8️⃣ Chunk Documents with `RecursiveCharacterTextSplitter`

We split all documents into smaller, overlapping chunks for better retrieval accuracy.

| Parameter | Value |
|-----------|-------|
| `chunk_size` | 800 characters |
| `chunk_overlap` | 100 characters |

In [ ]:
# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    length_function=len,
    is_separator_regex=False,
)

# Split all documents into chunks
all_chunks = text_splitter.split_documents(all_documents)

print(f"✂️  Total chunks created: {len(all_chunks)}")
print(f"\n{'='*60}")
print("📎 Sample Chunks (first 2):")
print(f"{'='*60}")

for i, chunk in enumerate(all_chunks[:2]):
    print(f"\n--- Chunk {i + 1} ---")
    print(f"Source  : {chunk.metadata.get('source', 'N/A')}")
    print(f"Type    : {chunk.metadata.get('type', 'N/A')}")
    print(f"Length  : {len(chunk.page_content)} chars")
    print(f"Content :\n{chunk.page_content[:500]}{'...' if len(chunk.page_content) > 500 else ''}")
    print()

---
## 9️⃣ Create HuggingFace Embeddings

We use the lightweight **`sentence-transformers/all-MiniLM-L6-v2`** model to generate dense vector embeddings for each chunk.

> This model runs locally — no API key required.

In [ ]:
# Initialize the embedding model (runs locally, no API key needed)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Quick sanity check — embed a test sentence
test_embedding = embedding_model.embed_query("Hello, world!")
print(f"✅ Embedding model loaded successfully.")
print(f"   Model       : sentence-transformers/all-MiniLM-L6-v2")
print(f"   Vector size : {len(test_embedding)} dimensions")

---
## 🔟 Store Chunks in ChromaDB

All chunked documents are embedded and stored in a persistent **ChromaDB** vector store.

| Setting | Value |
|---------|-------|
| `persist_directory` | `./chroma_db` |
| `collection_name` | `college_data` |

In [ ]:
# ChromaDB configuration
PERSIST_DIR = "./chroma_db"
COLLECTION_NAME = "college_data"

try:
    # Create the vector store from documents
    vectorstore = Chroma.from_documents(
        documents=all_chunks,
        embedding=embedding_model,
        persist_directory=PERSIST_DIR,
        collection_name=COLLECTION_NAME,
    )

    print(f"✅ ChromaDB vector store created successfully!")
    print(f"   📁 Persist directory : {os.path.abspath(PERSIST_DIR)}")
    print(f"   📦 Collection name   : {COLLECTION_NAME}")
    print(f"   🧩 Total vectors     : {vectorstore._collection.count()}")

except Exception as e:
    print(f"❌ Error creating ChromaDB vector store: {e}")

---
## 1️⃣1️⃣ Test Retrieval from ChromaDB

Let's verify that our vector store works by running a similarity search with a sample query.

In [ ]:
# Sample query to test retrieval
query = "What documents are required for admission?"
TOP_K = 3  # Number of similar chunks to retrieve

print(f"🔍 Query: \"{query}\"")
print(f"   Retrieving top {TOP_K} similar chunks...\n")

try:
    # Perform similarity search
    results = vectorstore.similarity_search(query, k=TOP_K)

    if not results:
        print("⚠️  No results found. The vector store may be empty.")
    else:
        for i, doc in enumerate(results, 1):
            print(f"{'='*60}")
            print(f"📌 Result {i}")
            print(f"{'='*60}")
            print(f"Source   : {doc.metadata.get('source', 'N/A')}")
            print(f"Type     : {doc.metadata.get('type', 'N/A')}")
            print(f"Content  :\n{doc.page_content}")
            print()

    print(f"✅ Retrieval test completed — {len(results)} result(s) returned.")

except Exception as e:
    print(f"❌ Error during retrieval: {e}")

---
## ✅ Pipeline Complete

The data ingestion pipeline is now finished. Here's a summary of what was accomplished:

| Step | Description |
|------|-------------|
| 📂 Load | `.txt`, `.pdf`, and `faq.json` files loaded with metadata |
| ✂️ Chunk | Documents split into 800-char chunks with 100-char overlap |
| 🧠 Embed | HuggingFace `all-MiniLM-L6-v2` embeddings generated |
| 💾 Store | All vectors persisted in ChromaDB (`./chroma_db`) |
| 🔍 Test | Similarity search verified with a sample query |

**Next step:** Connect this vector store to an LLM for answer generation in a separate notebook.